In [17]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
import os

In [31]:
# Step1: load raw pdf
DATA_PATH="data/"
def load_pdf_files(data):
    loader = DirectoryLoader(data, glob='*.pdf', loader_cls=PyPDFLoader)
    
    documents=loader.load()
    return documents

documents=load_pdf_files(data=DATA_PATH)

In [32]:
# Step2: Creaete chunks
def create_chunks(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", ".", "!", "?", " ", ""])
    # doc.page_content is the text content of each page in the PDF, doc in documents is a list of Document objects, 
    # and we are extracting the text content from each Document object to create chunks.
    chunks = text_splitter.split_documents(documents)
    print(f"Created {len(chunks)} text chunks")
    return chunks

text_chunks = create_chunks(documents)

Created 7080 text chunks


In [33]:
load_dotenv()

True

In [34]:
# Step3: Create Vector Embeddings
def get_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={
            'device': 'cpu',
            'trust_remote_code': True
        },
        encode_kwargs={
            'normalize_embeddings': True,
            'batch_size': 32
        }
    )
    return embeddings


embeddings = get_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4792.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
# Step4: Store embeddings in FAISS
DB_FAISS_PATH = "vectorestore/faiss_db"
os.makedirs(DB_FAISS_PATH, exist_ok=True)

if text_chunks:
	db = FAISS.from_documents(text_chunks, embeddings)
	db.save_local(DB_FAISS_PATH)
	print("Vector store successfully saved to: ", DB_FAISS_PATH)
else:
	print("No text chunks available to create vector store.")

KeyboardInterrupt: 

In [36]:
DB_FAISS_PATH="vectorstore/db_faiss"
os.makedirs(DB_FAISS_PATH, exist_ok=True)
db=FAISS.from_documents(text_chunks, embeddings)
db.save_local(DB_FAISS_PATH)